# Previous Behavior

In [1]:
import pandas as pd

In [2]:
# Mutating one object might accidentally mutate another
df = pd.DataFrame(
    {"foo": [1, 2, 3], "bar": [4, 5, 6]},
)
# Here, changing subset also updates the DataFrame
subset = df["foo"]
subset.iloc[0] = 100
df

,foo,bar
0,100,4
1,2,5
2,3,6


Copy-on-Write (CoW) solves accidentally modifying more than one object:
it explicitly <u>disallows</u> this.

In [ ]:
# #################################################
#        By enabling CoW, df is unchanged         #
# #################################################
pd.options.mode.copy_on_write = True

df = pd.DataFrame(
    {"foo": [1, 2, 3], "bar": [4, 5, 6]},
)
# Here, changing subset DOES NOT change the DataFrame
subset = df["foo"]
subset.iloc[0] = 100
df

,foo,bar
0,1,4
1,2,5
2,3,6


# Migrating to Copy-on-Write

##### Chained assignment will never work: Use loc instead

##### Accessing the underlying array of a pandas object, will return a read-only view

In [4]:
ser = pd.Series([1, 2, 3])
ser.to_numpy()

array([1, 2, 3])

##### Only one pandas object is updated at once

In [5]:
# Without CoW, both the df and subset would be updated
df = pd.DataFrame(
    {"foo": [1, 2, 3], "bar": [4, 5, 6]},
)
subset = df["foo"]
subset.iloc[0] = 100
# But since we have set: pd.options.mode.copy_on_write = True
# only the subset is updated and not the df.
df

,foo,bar
0,1,4
1,2,5
2,3,6


In [6]:
# Updating the column of the df with inplace, will also NOT work now.
df = pd.DataFrame(
    {"foo": [1, 2, 3], "bar": [4, 5, 6]},
)
df["foo"].replace(1, 5, inplace=True)
df

,foo,bar
0,1,4
1,2,5
2,3,6


In [7]:
# To achieve the replacement:
df = pd.DataFrame(
    {"foo": [1, 2, 3], "bar": [4, 5, 6]},
)
df.replace({"foo": {1: 5}}, inplace=True)
df

,foo,bar
0,5,4
1,2,5
2,3,6


In [8]:
# Or this one
df = pd.DataFrame(
    {"foo": [1, 2, 3], "bar": [4, 5, 6]},
)
df["foo"] = df["foo"].replace(1, 5)
df

,foo,bar
0,5,4
1,2,5
2,3,6


##### Constructors now copy NumPy arrays by default

# Description

In [9]:
# This example works
df = pd.DataFrame(
    {"foo": [1, 2, 3], "bar": [4, 5, 6]},
)
df.iloc[0, 0] = 100
df

,foo,bar
0,100,4
1,2,5
2,3,6


In [10]:
# This however, triggers a copy of the data under CoW:
df = pd.DataFrame(
    {"foo": [1, 2, 3], "bar": [4, 5, 6]},
)
df2 = df.reset_index(drop=True)
df2.iloc[0, 0] = 100
df

,foo,bar
0,1,4
1,2,5
2,3,6


In [11]:
df2

,foo,bar
0,100,4
1,2,5
2,3,6


In [12]:
df = pd.DataFrame(
    {"foo": [1, 2, 3], "bar": [4, 5, 6]},
)
df = df.reset_index(drop=True)
df.iloc[0, 0] = 100
df

,foo,bar
0,100,4
1,2,5
2,3,6


In [13]:
# Without CoW, the view AND the parent object were modified
with pd.option_context("mode.copy_on_write", False):
    df = pd.DataFrame({"foo": [1, 2, 3], "bar": [4, 5, 6]})
    view = df[:]
    df.iloc[0, 0] = 100
df

,foo,bar
0,100,4
1,2,5
2,3,6


In [14]:
view

,foo,bar
0,100,4
1,2,5
2,3,6


# Chained Assignment
Updating an object through 2 subsequent indexing operations

In [ ]:
with pd.option_context("mode.copy_on_write", False):
    df = pd.DataFrame({"foo": [1, 2, 3], "bar": [4, 5, 6]})
    # Update column foo where column bar is greater than 5
    # With CoW enabled, this would generate a ChainedAssignmentError
    df["foo"][df["bar"] > 5] = 100
df

,foo,bar
0,1,4
1,2,5
2,100,6


In [21]:
df = pd.DataFrame({"foo": [1, 2, 3], "bar": [4, 5, 6]})
df.loc[df["bar"] > 5, "foo"] = 100
df

,foo,bar
0,1,4
1,2,5
2,100,6


# Read-only NumPy Arrays

In [23]:
df = pd.DataFrame(
    {"a": [1, 2], "b": [1.5, 2.5]},
)
df.to_numpy()

array([[1. , 1.5],
       [2. , 2.5]])

In [24]:
arr = df.to_numpy()
arr[0, 0] = 100

In [26]:
arr = df.to_numpy()
arr.flags.writeable = True
arr[0, 0] = 100
arr

array([[100. ,   1.5],
       [  2. ,   2.5]])